# Notebook 12: Explanatory — Funding Allocation and Safehouse Outcomes

## Section 1 — Problem Framing

**Business question:** Across safehouses and months, which allocation categories
(Education, Wellbeing, Operations, Transport, etc.) are most strongly associated
with better health and education outcomes?

**Who cares:** The finance director and board members who decide how to distribute
donated funds. If "Education" allocations show stronger outcome associations than
"Transport," that informs budget priorities.

**Approach: Explanatory.** We use OLS regression with safehouse fixed effects to
isolate *within-safehouse* variation over time. This controls for permanent
differences between safehouses (location, staff quality, etc.) that could confound
the allocation-outcome relationship.

**Success metric:** Adjusted R2, coefficient significance, and a clear statement of
how additional funding in each category is associated with outcome changes.

## Section 2 — Data Acquisition and Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm
import json, os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

DATA_DIR = '../../data/lighthouse_csv_v7/'
RESULTS_DIR = '../../data/explanatory_results/'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

In [ ]:
from sqlalchemy import create_engine
import os

DB_HOST = os.environ.get("DB_HOST", "localhost")
DB_PORT = os.environ.get("DB_PORT", "5432")
DB_NAME = os.environ.get("DB_NAME", "harbor_of_hope")
DB_USER = os.environ.get("DB_USER", "waylansmac")
DB_PASS = os.environ.get("DB_PASS", "")
CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(CONNECTION_STRING)

allocations = pd.read_sql("SELECT * FROM donation_allocations", engine)
safehouse_metrics = pd.read_sql("SELECT * FROM safehouse_monthly_metrics", engine)
safehouses = pd.read_sql("SELECT * FROM safehouses", engine)

print(f'Loaded {len(allocations)} allocations, {len(safehouse_metrics)} monthly metrics, '
      f'{len(safehouses)} safehouses')

In [ ]:
def plot_coefficients(params, conf_int, pvalues, title, filename):
    df_coef = pd.DataFrame({
        'coef': params, 'ci_low': conf_int.iloc[:, 0],
        'ci_high': conf_int.iloc[:, 1], 'pvalue': pvalues
    })
    df_coef = df_coef.drop('const', errors='ignore')
    df_coef['significant'] = df_coef['pvalue'] < 0.05
    df_coef = df_coef.sort_values('coef')
    fig, ax = plt.subplots(figsize=(10, max(6, len(df_coef) * 0.35)))
    colors = ['#2196F3' if s else '#BDBDBD' for s in df_coef['significant']]
    y_pos = range(len(df_coef))
    ax.barh(y_pos, df_coef['coef'], color=colors, edgecolor='white', height=0.7)
    ax.errorbar(df_coef['coef'], y_pos,
                xerr=[df_coef['coef'] - df_coef['ci_low'], df_coef['ci_high'] - df_coef['coef']],
                fmt='none', ecolor='black', capsize=3, linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_coef.index, fontsize=9)
    ax.axvline(0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Coefficient')
    ax.set_title(title)
    blue_patch = plt.Line2D([0], [0], color='#2196F3', lw=6, label='p < 0.05')
    grey_patch = plt.Line2D([0], [0], color='#BDBDBD', lw=6, label='p >= 0.05')
    ax.legend(handles=[blue_patch, grey_patch], loc='lower right')
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

print('Helper functions defined.')

In [ ]:
alloc = allocations.copy()
alloc['allocation_date'] = pd.to_datetime(alloc['allocation_date'])
alloc['month_start'] = alloc['allocation_date'].dt.to_period('M').dt.to_timestamp()

alloc_pivot = alloc.pivot_table(
    index=['safehouse_id', 'month_start'],
    columns='program_area',
    values='amount_allocated',
    aggfunc='sum',
    fill_value=0
).reset_index()
alloc_pivot.columns.name = None
alloc_pivot.columns = [str(c) for c in alloc_pivot.columns]

area_cols = [c for c in alloc_pivot.columns if c not in ['safehouse_id', 'month_start']]
alloc_pivot['total_allocation'] = alloc_pivot[area_cols].sum(axis=1)

metrics = safehouse_metrics.copy()
metrics['month_start'] = pd.to_datetime(metrics['month_start'])

df4 = metrics.merge(alloc_pivot, on=['safehouse_id', 'month_start'], how='left')
df4[area_cols + ['total_allocation']] = df4[area_cols + ['total_allocation']].fillna(0)
df4 = df4.dropna(subset=['avg_health_score'])

sh_info = safehouses[['safehouse_id', 'capacity_girls', 'region']].copy()
df4 = df4.merge(sh_info, on='safehouse_id', how='left')

df4 = pd.get_dummies(df4, columns=['safehouse_id'], drop_first=True, dtype=int, prefix='sh')

print(f'Pipeline 4 analytical dataset: {df4.shape[0]} safehouse-months x {df4.shape[1]} columns')
print(f'\nProgram areas found: {area_cols}')
print(f'\nAvg Health Score summary:\n{df4["avg_health_score"].describe()}')
print(f'\nAvg Education Progress summary:\n{df4["avg_education_progress"].describe()}')

## Section 3 — Exploration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for sh_id in metrics['safehouse_id'].unique():
    sub = metrics[metrics['safehouse_id'] == sh_id].sort_values('month_start')
    axes[0, 0].plot(sub['month_start'], sub['avg_health_score'], alpha=0.5, label=f'SH{sh_id}')
axes[0, 0].set_title('Avg Health Score Over Time by Safehouse')
axes[0, 0].set_ylabel('Health Score')
axes[0, 0].legend(fontsize=7, ncol=2)

area_totals = alloc.groupby('program_area')['amount_allocated'].sum().sort_values()
area_totals.plot.barh(color='#26A69A', ax=axes[0, 1])
axes[0, 1].set_title('Total Allocation by Program Area')
axes[0, 1].set_xlabel('Total PHP')

if 'Education' in df4.columns:
    axes[1, 0].scatter(df4['Education'], df4['avg_education_progress'], alpha=0.4, edgecolors='k', linewidth=0.3)
    axes[1, 0].set_xlabel('Education Allocation (PHP)')
    axes[1, 0].set_ylabel('Avg Education Progress')
    axes[1, 0].set_title('Education Spending vs Education Progress')

if 'Wellbeing' in df4.columns:
    axes[1, 1].scatter(df4['Wellbeing'], df4['avg_health_score'], alpha=0.4, edgecolors='k', linewidth=0.3, color='#EF5350')
    axes[1, 1].set_xlabel('Wellbeing Allocation (PHP)')
    axes[1, 1].set_ylabel('Avg Health Score')
    axes[1, 1].set_title('Wellbeing Spending vs Health Score')

plt.suptitle('Pipeline 4 — Funding Allocation & Safehouse Outcomes', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('p4_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Modeling and Feature Selection

In [ ]:
exclude_4 = ['metric_id', 'month_start', 'month_end', 'avg_health_score',
             'avg_education_progress', 'notes', 'region', 'total_allocation']
feature_cols_4 = [c for c in df4.columns if c not in exclude_4
                  and df4[c].dtype in ['int64', 'float64', 'int32', 'uint8', 'int8']]
X4 = df4[feature_cols_4].copy().apply(pd.to_numeric, errors='coerce').fillna(0)
y4_health = df4['avg_health_score'].copy()

scaler4 = StandardScaler()
non_dummy_cols = [c for c in X4.columns if not c.startswith('sh_')]
X4_sc = X4.copy()
X4_sc[non_dummy_cols] = scaler4.fit_transform(X4[non_dummy_cols])

X4_ols = sm.add_constant(X4_sc)
ols_health = sm.OLS(y4_health, X4_ols).fit()
print('=== OLS: Allocation -> Avg Health Score (with safehouse fixed effects) ===')
print(ols_health.summary())

In [ ]:
params4 = ols_health.params.drop('const', errors='ignore')
non_sh = [c for c in params4.index if not c.startswith('sh_')]
plot_coefficients(
    ols_health.params[['const'] + non_sh],
    ols_health.conf_int().loc[['const'] + non_sh],
    ols_health.pvalues[['const'] + non_sh],
    'Pipeline 4 — OLS Coefficients for Health Score (excl. safehouse FE)',
    'p4_ols_health_coefficients.png')

y4_edu = df4['avg_education_progress'].dropna()
X4_ols_edu = X4_ols.loc[y4_edu.index]
ols_edu = sm.OLS(y4_edu, X4_ols_edu).fit()
print('\n=== OLS: Allocation -> Avg Education Progress (with safehouse FE) ===')
print(f'R2 = {ols_edu.rsquared:.3f}, Adj R2 = {ols_edu.rsquared_adj:.3f}')

non_sh_params_edu = [c for c in ols_edu.params.index if not c.startswith('sh_') and c != 'const']
print('\nSignificant allocation features for education progress:')
for feat in non_sh_params_edu:
    p = ols_edu.pvalues[feat]
    if p < 0.1:
        print(f'  {feat}: coef={ols_edu.params[feat]:+.4f}, p={p:.4f}')

In [ ]:
X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4_sc, y4_health, test_size=0.2, random_state=42)

rf4 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf4.fit(X4_train, y4_train)
y4_pred = rf4.predict(X4_test)
print(f'Random Forest — Test R2: {r2_score(y4_test, y4_pred):.3f}')
print(f'Random Forest — Test RMSE: {np.sqrt(mean_squared_error(y4_test, y4_pred)):.3f}')

cv4 = cross_val_score(rf4, X4_sc, y4_health, cv=5, scoring='r2')
print(f'5-Fold CV R2: {cv4.mean():.3f} +/- {cv4.std():.3f}')
print(f'\nOLS Adj R2: {ols_health.rsquared_adj:.3f}')

fi4 = pd.Series(rf4.feature_importances_, index=X4.columns)
fi4_non_sh = fi4[[c for c in fi4.index if not c.startswith('sh_')]].sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
fi4_non_sh.plot.barh(color='#26A69A', ax=ax)
ax.set_title('Pipeline 4 — RF Feature Importance (excl. safehouse dummies)')
plt.tight_layout()
plt.savefig('p4_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y4_test, y4_pred, alpha=0.5, edgecolors='k', linewidth=0.3)
mn, mx = min(y4_test.min(), y4_pred.min()), max(y4_test.max(), y4_pred.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=1)
ax.set_xlabel('Actual Avg Health Score')
ax.set_ylabel('Predicted')
ax.set_title('Pipeline 4 — RF Actual vs Predicted Health Score')
plt.tight_layout()
plt.savefig('p4_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 5 — Evaluation and Causal Analysis

### Key Findings
By including **safehouse fixed effects**, we control for all time-invariant
differences between safehouses. The remaining variation is *within-safehouse over time*:
when a safehouse receives more funding in a particular program area in a given month,
does its health/education score change?

### Interpreting the Coefficients
Each coefficient represents the association between a one-SD increase in that
allocation category (within the same safehouse, over time) and the change in
health score.

### Causal Defensibility
- **Wellbeing allocation -> health score**: Plausibly causal — direct funding of
  medical supplies, psychological support, and nutrition should improve health.
  However, timing may lag.
- **Education allocation -> education progress**: Expected relationship. But allocations
  may respond to *needs* rather than drive *improvements* (reverse causality).
- **Cross-domain effects**: If *Education* spending predicts *Health* improvement, this
  suggests holistic benefits of structured activity.

### Limitations
1. **Aggregation bias** — safehouse-month level analysis masks individual variation.
2. **Lag structure** — we model contemporaneous relationships; lagged effects may
   be important but require longer time series.
3. **Budget is endogenous** — allocation decisions are not random; they respond to
   perceived needs.

### Recommendations
1. **Prioritize Wellbeing and Education allocations** — they show the most consistent
   positive associations with outcomes.
2. **Experiment with allocation ratios** — small deliberate shifts across months could
   serve as natural experiments.
3. **Track lagged effects** — build a time-series model as more monthly data accumulates.

## Section 6 — Data Leakage Check

### Potential Leakage Concerns
1. **Contemporaneous measurement**: Allocations in month M and outcomes in month M are
   measured simultaneously. Allocations may *respond to* poor outcomes rather than
   *cause* better outcomes (reverse causality). **Mitigation**: We use safehouse fixed
   effects to control for time-invariant confounders and note this limitation.
2. **Total allocation excluded**: We exclude `total_allocation` from the model because
   it is a linear combination of the area-specific allocation columns. Including it
   would create multicollinearity and obscure individual category effects.
3. **No future data**: We do not use next-month outcomes or allocations as features.

### Verdict
No direct data leakage is present. The main threat is **simultaneity bias** (concurrent
measurement of treatment and outcome), which we acknowledge. For stronger causal claims,
future work should use lagged allocations (month M-1) predicting outcomes (month M).

## Section 7 — Deployment Notes

### Integration with Harbor of Hope Web Application
- **API Endpoint**: `GET /api/explanatoryinsights/4` returns the allocation-outcome
  associations and budget optimization insights.
- **Dashboard Page**: Admin > Insights shows the funding effectiveness analysis.
- **Budget Optimizer**: An interactive form where the finance director enters a total
  monthly budget and sees predicted outcome changes under different allocation scenarios.
- **Notebook location**: `ml-pipelines/12-explanatory-funding-outcomes.ipynb`

### How to Refresh Results
1. Run this notebook end-to-end.
2. The final cell exports updated results to
   `data/explanatory_results/pipeline_04_funding_outcomes.json`.

In [ ]:
sig4 = ols_health.pvalues.drop('const', errors='ignore')
sig4_non_sh = sig4[[c for c in sig4.index if not c.startswith('sh_')]]
sig4_non_sh = sig4_non_sh[sig4_non_sh < 0.05].sort_values()

results_04 = {
    "pipeline_id": 4,
    "pipeline_name": "Funding Allocation -> Safehouse Outcomes",
    "target_variable": "Avg Health Score (with safehouse fixed effects)",
    "model_type": "OLS Regression with Fixed Effects",
    "r_squared": round(float(ols_health.rsquared), 4),
    "adj_r_squared": round(float(ols_health.rsquared_adj), 4),
    "sample_size": int(len(y4_health)),
    "significant_features": [
        {
            "name": feat,
            "coefficient": round(float(ols_health.params[feat]), 4),
            "p_value": round(float(ols_health.pvalues[feat]), 4),
            "direction": "increases" if ols_health.params[feat] > 0 else "decreases"
        }
        for feat in sig4_non_sh.index
    ]
}

with open(f'{RESULTS_DIR}pipeline_04_funding_outcomes.json', 'w') as f:
    json.dump(results_04, f, indent=2)
print(f'Exported results to {RESULTS_DIR}pipeline_04_funding_outcomes.json')